# Staggered Federated Learning with PyTorch (DenseNet-121) on Rhino FCP

This notebook runs the `pytorch-densenet/scatter-and-gather` example on Rhino's Federated Computing Platform (FCP): N sites that share one GPU agent train a DenseNet-121 pneumonia classifier with NVFlare's ScatterAndGather workflow and the in-process client, taking turns on the GPU (at most 4 at a time). It makes the data shards, creates the data schema, registers one dataset per site, creates the Code Object and runs the training.

See `README.md` in this folder for what the example does and how to set it up, and `../../README.md` for the staggered FL guide.

#### Before you run this notebook
1. Install `rhino_health` (2.1.25 or newer, for `CodeTypes.NVIDIA_FLARE_V2_8`) and `pandas`: the first cell under Setup does it
2. A project with N collaborating workgroups whose sites run on the same GPU agent (README, "Running this example on FCP", step 1)
3. The Kaggle chest X-ray archive downloaded; no need to unzip it (README step 2)
4. The image built and pushed (README step 3)

## Setup

In [1]:
%pip install "rhino_health>=2.1.25" pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import math
import os
from getpass import getpass

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    ModelTrainInput,
)

# Pin the working directory to this notebook's folder so relative paths resolve
# regardless of where the notebook is opened from (e.g. VS Code, Jupyter Lab, etc.)
os.chdir(os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', 'notebook.ipynb'))))
print("Setup Complete")

Setup Complete


## Container image

Paste the URI that `docker-push.sh` printed (README step 3).

In [3]:
CONTAINER_IMAGE_URI = "<YOUR_CONTAINER_IMAGE_URI>"  # UPDATE THIS TO THE URI PRINTED BY docker-push.sh

if CONTAINER_IMAGE_URI.startswith("<"):
    raise ValueError("Set CONTAINER_IMAGE_URI to the URI printed by docker-push.sh before continuing.")
print("Completed.")

Completed.


## Environment

The FCP environment to work with: its API URL ends in `/api/`. `ApiEnvironment` has the Rhino-hosted ones, e.g. `ApiEnvironment.PROD_AWS_URL` (`https://prod.rhinohealth.com/api/`) or `ApiEnvironment.PROD_GCP_URL` (`https://prod.rhinofcp.com/api/`). For any other environment, paste its URL as a string, e.g. `"https://dev3.rhinofcp.com/api/"`.

**Variables to adjust:**
- `RHINO_API_URL` - the API URL of the FCP environment you log in to

In [4]:
from rhino_health import ApiEnvironment

RHINO_API_URL = ApiEnvironment.PROD_AWS_URL  # e.g. ApiEnvironment.PROD_GCP_URL, or "https://dev3.rhinofcp.com/api/"
print(f"Using {RHINO_API_URL}")

Using https://dev3.rhinohealth.com/api/


## Authentication

Log in to FCP; you'll be prompted for your password.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [9]:
USERNAME = "<YOUR_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

session = rh.login(username=USERNAME, password=getpass(), rhino_api_url=RHINO_API_URL)
print("Logged In")

 ········


You are not using the latest version of the Rhino SDK.
Latest version: 2.2.8 (requires Python >=3.10)
Current version: 2.2.7 (running Python 3.14.5)
To upgrade, run: pip install --upgrade rhino_health

Note: If pip reports the requirement is already satisfied, then it it likely that the Python version you are running is no longer supported by the latest SDK release. In that case, upgrade to a newer version of Python supported by the latest release of the Rhino SDK.
Logged In


## Project and site workgroups

FCP runs one federated learning site per dataset, and each workgroup holds one dataset here. This finds your project and its collaborating workgroups whose name starts with `SITE_WORKGROUP_PREFIX`: these are the N sites.

**Variables to adjust:**
- `PROJECT_NAME` - the project your workgroups collaborate in
- `SITE_WORKGROUP_PREFIX` - the common prefix of the site workgroups' names (e.g. `stagger-site-` for `stagger-site-01` ... `stagger-site-08`)
- `NUM_SITES` - optional: use only the first N matching workgroups (by name); `None` uses all of them

In [6]:
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
SITE_WORKGROUP_PREFIX = "<YOUR_SITE_WORKGROUP_PREFIX>"
NUM_SITES = None

project = session.project.get_project_by_name(PROJECT_NAME)
if project is None:
    raise ValueError(f"Project {PROJECT_NAME!r} not found.")
site_workgroups = sorted(
    (workgroup for workgroup in project.collaborating_workgroups if workgroup.name.startswith(SITE_WORKGROUP_PREFIX)),
    key=lambda workgroup: workgroup.name,
)[:NUM_SITES]
if not site_workgroups:
    raise ValueError(f"No collaborating workgroups starting with {SITE_WORKGROUP_PREFIX!r} in project {PROJECT_NAME!r}.")
for workgroup in site_workgroups:
    print(f"{workgroup.name} ({workgroup.uid})")
print(f"{len(site_workgroups)} sites")

sfl-1021-site-01 (bc27b658-05fe-4c25-bd85-6601907548fc)
sfl-1021-site-02 (766be1b7-490a-4a3b-b5b8-e9dced1c5c02)
sfl-1021-site-03 (05c9f701-3b38-496d-95e7-ec3ea04161b6)
sfl-1021-site-04 (74952f15-adef-42af-93e9-125fd6a216c7)
sfl-1021-site-05 (6f12ee3e-591d-4e19-bf05-2b747259955a)
sfl-1021-site-06 (233ee33e-ea00-45ae-a75e-3aceb3427caf)
sfl-1021-site-07 (effa56de-25a3-4c62-96e7-7de19fffe0c1)
sfl-1021-site-08 (bcbd933d-3b7d-4667-aa85-7c58ef30670e)
8 sites


## Data shards

Takes about 150 images per site from the Kaggle archive and splits them by patient: the images go to `../data/images/` under unique names, and `../data/shards/shard_01.csv` ... `shard_NN.csv` list each site's images, with the columns `JPG_file`, `Patient_ID` and `Pneumonia`. There is no need to unzip the archive. To use more of the data, add `"--images-per-site", "<n>"` to the command (`0` uses all 5,856 images); training time per round doesn't change, because each round trains a fixed number of steps.

**Then put the data where the GPU agent can read it.** The notebook can't do this step, and the files must be there before the datasets are registered below. Copy the `../data/images` and `../data/shards` folders, side by side in one folder, to either:
- the agent's own storage (the on-prem path your agent imports datasets from), or
- a cloud bucket connected to the agent as Client Mounted Storage.

The folder must look like this; set `DATA_LOCATION` under "Datasets" to its path as the agent sees it:
```
<DATA_LOCATION>/
├── images/      all the .jpeg files, flat (no sub-folders)
│   ├── test_NORMAL_IM-0001-0001.jpeg
│   └── ...
└── shards/      one CSV per site
    ├── shard_01.csv
    └── ...
```
Each shard's `JPG_file` column holds a file name inside `images/`, so keep the two folders together and don't rename the images.

**Variables to adjust:**
- `ARCHIVE` - the downloaded Kaggle `.zip` (or the folder you unzipped it into)

To re-run with other settings, first delete `../data/images` and `../data/shards`: the cell adds to what is there.

In [11]:
import subprocess
import sys

ARCHIVE = "../data/archive.zip"

subprocess.run(
    [sys.executable, "../data/make_shards.py", "--archive", ARCHIVE, "--num-shards", str(len(site_workgroups))],
    check=True,
)

Wrote 8 shards, 1205 images, to /Users/arye/dev/user-resources/examples/nvflare/staggered-fl/pytorch-densenet/data
       images  patients  pneumonia_rate
shard                                  
1         150       101           0.640
2         150        78           0.767
3         151        91           0.656
4         150        80           0.773
5         151        81           0.735
6         153        98           0.660
7         150        84           0.687
8         150        94           0.687


CompletedProcess(args=['/opt/homebrew/Cellar/jupyterlab/4.5.7_3/libexec/bin/python', '../data/make_shards.py', '--archive', '../data/archive.zip', '--num-shards', '8'], returncode=0)

## Data schema

Creates the data schema for the shards (columns `JPG_file`, `Patient_ID`, `Pneumonia`) in your project, owned by the project's primary workgroup. `JPG_file` is a `Filename` column, so the import copies each listed image into the site's dataset. If a schema with this name already exists in the project, it is reused.

In [8]:
from rhino_health.lib.endpoints.data_schema.data_schema_dataclass import DataSchemaCreateInput

DATA_SCHEMA_NAME = "Chest X-Ray Pneumonia - Staggered FL"
DATA_SCHEMA_FIELDS = [
    "Variable Name,JPG_file,Patient_ID,Pneumonia\n",
    "Identifier,JPG_file,Patient_ID,Pneumonia\n",
    "Description,Chest X-ray image file name,Patient the image belongs to,Pneumonia (1) or normal (0)\n",
    "Role,Input,Input,Output\n",
    "Type,Filename,String,Integer\n",
    "Type Parameters,,,\n",
    "Units,,,\n",
    "May Contain PHI?,FALSE,FALSE,FALSE\n",
    "Permissions,Default,Default,Default\n",
]

data_schema = session.project.get_data_schema_by_name(DATA_SCHEMA_NAME, project_uid=project.uid)
if data_schema is None:
    data_schema = session.data_schema.create_data_schema(
        DataSchemaCreateInput(
            name=DATA_SCHEMA_NAME,
            description="Chest X-ray images and whether they show pneumonia",
            primary_workgroup_uid=project.primary_workgroup_uid,
            project_uid=project.uid,
            schema_fields=DATA_SCHEMA_FIELDS,
        )
    )
print(f"Data schema: {data_schema.name} ({data_schema.uid})")

Data schema: Chest X-Ray Pneumonia - Staggered FL -- Validation (e0c682ca-7c6a-4008-8343-ec31f2833f41)


## Datasets

Registers shard *i* as a dataset in the *i*-th site workgroup, acting as that workgroup, so every site gets its own share of the data. Datasets that already exist are reused, so the cell is safe to re-run. It then lists every dataset in the project whose name starts with `DATASET_NAME_PREFIX`: one per site.

Registering as a workgroup needs you to be a member of it. If you are not a member of every site workgroup, set `REGISTER_DATASETS = False` and register the shards in the FCP UI instead, once per workgroup (**Datasets → Add Dataset**, the data schema above, the names `cxr-shard-01` ... in workgroup order, `shards/shard_NN.csv` as the CSV and `images` as the files folder).

**Variables to adjust:**
- `DATA_LOCATION` - the path on the GPU agent where you copied `../data/images` and `../data/shards`
- `DATASET_NAME_PREFIX` - the common prefix of the shard datasets
- `REGISTER_DATASETS` - `False` to only list datasets you registered in the UI

In [13]:
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput

DATA_LOCATION = "<PATH_TO_IMAGES_AND_SHARDS_ON_THE_AGENT>"
DATASET_NAME_PREFIX = "cxr-shard-"
REGISTER_DATASETS = True

if REGISTER_DATASETS:
    for index, workgroup in enumerate(site_workgroups, start=1):
        session.switch_workgroup(workgroup)
        try:
            dataset = session.dataset.add_dataset(
                DatasetCreateInput(
                    name=f"{DATASET_NAME_PREFIX}{index:02d}",
                    description=f"Chest X-ray shard {index} of {len(site_workgroups)}",
                    project_uid=project.uid,
                    workgroup_uid=workgroup.uid,
                    data_schema_uid=data_schema.uid,
                    csv_filesystem_location=f"{DATA_LOCATION}/shards/shard_{index:02d}.csv",
                    file_base_path=f"{DATA_LOCATION}/images",
                    method="filesystem",
                    is_data_deidentified=True,
                )
            )
        finally:
            session.switch_workgroup(session.current_user.primary_workgroup)
        print(f"{workgroup.name}: {dataset.name} ({dataset.import_status})")
        if dataset.import_status == "Error":
            raise RuntimeError(f"Importing {dataset.name} failed: check that {DATA_LOCATION}/shards and {DATA_LOCATION}/images exist on the agent.")

datasets = sorted(
    (dataset for dataset in session.project.get_datasets(project.uid) if dataset.name.startswith(DATASET_NAME_PREFIX)),
    key=lambda dataset: dataset.name,
)
if len(datasets) != len(site_workgroups):
    raise ValueError(f"Found {len(datasets)} datasets starting with {DATASET_NAME_PREFIX!r}, expected one per site ({len(site_workgroups)}).")
for dataset in datasets:
    print(f"{dataset.name} ({dataset.uid})")

sfl-1021-site-01: cxr-shard-01 (Complete)
sfl-1021-site-02: cxr-shard-02 (Complete)
sfl-1021-site-03: cxr-shard-03 (Complete)
sfl-1021-site-04: cxr-shard-04 (Complete)
sfl-1021-site-05: cxr-shard-05 (Complete)
sfl-1021-site-06: cxr-shard-06 (Complete)
sfl-1021-site-07: cxr-shard-07 (Complete)
sfl-1021-site-08: cxr-shard-08 (Complete)
cxr-shard-01 (b3bcd173-8c4a-49bb-935b-540398d080d9)
cxr-shard-02 (a9a31ab0-caf9-4401-9d81-6ca0651c4e2b)
cxr-shard-03 (159645b3-a5f2-4532-b2c9-ac2103e87ceb)
cxr-shard-04 (675865de-6dd4-4ee3-bbee-d1b0da1747a0)
cxr-shard-05 (c50e7f7f-3d2f-40b6-b1ed-3c402b5c88d1)
cxr-shard-06 (b38854e5-8a04-47d5-a0e5-c873695ff375)
cxr-shard-07 (312ce836-558c-479b-95e3-0fe6bdc52115)
cxr-shard-08 (8e622ca2-c04c-49f2-9b88-9f0e3115afea)


## NVFlare Code Object Creation

The Code Object points at the image you pushed. `return_existing=False, add_version_if_exists=True` makes a new version when one with this name exists, so a new image URI takes effect on re-runs.

In [15]:
code_object_input = CodeObjectCreateInput(
    name="Staggered FL - PyTorch DenseNet",
    description="DenseNet-121 chest X-ray pneumonia classifier, sites take turns on a shared GPU",
    input_data_schema_uids=[datasets[0].data_schema_uid],
    output_data_schema_uids=[None],
    project_uid=project.uid,
    code_type=CodeTypes.NVIDIA_FLARE_V2_8,  # must match requirements.txt: nvflare==2.8.1
    config={"container_image_uri": CONTAINER_IMAGE_URI},
)
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
code_object = code_object.wait_for_build()  # returns immediately - the image is already built
print(f"Created code object: {code_object.name} ({code_object.uid})")

Created code object: Staggered FL - PyTorch DenseNet -- VALIDATION (bbffafe6-45c4-4d47-826a-ab95c20405f7)


## Job configuration

Reads the two configs, removes their `//` comment lines (JSON doesn't allow comments), and fits them to your datasets:
- `min_clients` is set to the number of datasets: FCP starts the job only when every site is up.
- `min_successful_clients` (sites that must succeed in each round) is kept at most `min_clients`.
- `timeout_seconds`, after which FCP stops the whole run, is computed from the rounds, the sites and K, because sites take turns.

**Variables to adjust:**
- `LOCAL_TRAINING_SECONDS` - how long one site trains per round when K sites train at once (~80 s for this example on an NVIDIA L4)
- `OVERHEAD_SECONDS` - time per turn outside training (~15 s here with the in-process client, including validating the global model)

In [16]:
LOCAL_TRAINING_SECONDS = 80
OVERHEAD_SECONDS = 15
SAFETY_FACTOR = 1.3


def load_config(path):
    """Read a commented example config and return it as a dict."""
    with open(path) as f:
        return json.loads("".join(line for line in f if not line.lstrip().startswith("//")))


config_fed_client = load_config("app/config/config_fed_client.json")
config_fed_server = load_config("app/config/config_fed_server.json")

workflow_args = config_fed_server["workflows"][0]["args"]
num_sites = len(datasets)
workflow_args["min_clients"] = num_sites
workflow_args["min_successful_clients"] = min(workflow_args.get("min_successful_clients", num_sites), num_sites)

waves_per_round = math.ceil(num_sites / workflow_args["max_concurrent"])
timeout_seconds = math.ceil(
    workflow_args["num_rounds"] * waves_per_round * (LOCAL_TRAINING_SECONDS + OVERHEAD_SECONDS) * SAFETY_FACTOR
)
timeout_seconds = max(timeout_seconds, 1800)  # leaves room for start-up on small jobs
print(
    f"{num_sites} sites, K = {workflow_args['max_concurrent']} ({waves_per_round} waves per round), "
    f"{workflow_args['num_rounds']} rounds, min_successful_clients = {workflow_args['min_successful_clients']}, "
    f"timeout_seconds = {timeout_seconds}"
)

8 sites, K = 4 (2 waves per round), 3 rounds, min_successful_clients = 7, timeout_seconds = 1800


## Federated training run

Starts the run on every dataset (one site each) and waits for it to finish.

In [17]:
run_params = ModelTrainInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[dataset.uid for dataset in datasets],
    one_fl_client_per_dataset=True,
    validation_dataset_uids=[],
    validation_datasets_inference_suffix="",
    timeout_seconds=timeout_seconds,
    config_fed_client=json.dumps(config_fed_client),
    config_fed_server=json.dumps(config_fed_server),
)

model_train = session.code_object.train_model(run_params)
code_run = model_train.wait_for_completion(timeout_seconds, poll_frequency=30)
print(f"Training finished with status: {code_run.status}")

Waiting for code run to complete (0 hours 0 minutes and a second)
Waiting for code run to complete (0 hours 0 minutes and 31 seconds)
Waiting for code run to complete (0 hours a minute and 2 seconds)
Waiting for code run to complete (0 hours a minute and 32 seconds)
Waiting for code run to complete (0 hours 2 minutes and 3 seconds)
Waiting for code run to complete (0 hours 2 minutes and 33 seconds)
Waiting for code run to complete (0 hours 3 minutes and 4 seconds)
Waiting for code run to complete (0 hours 3 minutes and 35 seconds)
Waiting for code run to complete (0 hours 4 minutes and 5 seconds)
Waiting for code run to complete (0 hours 4 minutes and 36 seconds)
Waiting for code run to complete (0 hours 5 minutes and 7 seconds)
Waiting for code run to complete (0 hours 5 minutes and 37 seconds)
Waiting for code run to complete (0 hours 6 minutes and 8 seconds)
Waiting for code run to complete (0 hours 6 minutes and 39 seconds)
Waiting for code run to complete (0 hours 7 minutes and 9 

In [ ]:
# If the run failed, its errors are here. The server log (FCP UI, the run's logs) has a "Site ... failed" or
# "Job stopped ..." line explaining which sites failed and why; see "Failures" in ../../README.md.
print(code_run.errors)

## Check the staggering

Saves the run's server log to `server.log` (the FCP UI shows the logs but can't download them) and runs `../../stagger_report.py` on it. The report prints one line per round (duration, failed sites, the most sites that trained at once per agent) and ends with `OK` when no agent went over K.

In [ ]:
logs = session.code_run.get_logs(code_run.uid)
with open("server.log", "w") as f:
    f.write("\n".join(str(entry.get("logs", "")) for entry in logs.logs))

subprocess.run([sys.executable, "../../stagger_report.py", "server.log"])

## Model Parameters Download

Downloads the trained global model. You can also download it from the FCP UI: Code Runs, the 3 dots next to the run, "Download Model Parameters".

In [ ]:
weights = session.code_run.get_model_params(code_run.uid)
with open("model_parameters.pt", "wb") as f:
    f.write(weights.getbuffer())
print("Saved model to model_parameters.pt")

## Cleanup

Removes the Code Object this notebook created. Your project, workgroups, data schema and datasets are left as they are, and so are the shards on the agent and the pushed image.

In [ ]:
CLEANUP = False  # Set to True to delete the Code Object created above

if CLEANUP:
    session.code_object.remove_code_object(code_object)
    print("Removed the code object from FCP.")
else:
    print("Skipped - set CLEANUP = True above to remove the Code Object this notebook created.")

## Additional Resources

- [Staggered FL guide](../../README.md)
- [Rhino SDK Documentation](https://rhinohealth.github.io/rhino_sdk_docs/html/autoapi/index.html)
- [Rhino FCP Platform Documentation](https://docs.rhinofcp.com/)
- [NVFlare GitHub](https://github.com/NVIDIA/NVFlare/tree/main)